### Make a temporary space in /flash

In [1]:
import os, random, string

In [2]:
cwd = os.path.basename(os.getcwd())
tmpdir = os.path.join("/flash/ReiterU/tmp/", 'tomo_' + ''.join(random.choices(string.ascii_uppercase, k=12)))
print(tmpdir)
if not os.path.exists(tmpdir):
    os.makedirs(tmpdir)

/flash/ReiterU/tmp/tomo_LVFLQLRVBQBF


### PARAMETERS

In [3]:
#### path to relevant files
rootdir = "/bucket/ReiterU/OLdev/experiments/20250521_squid_cal520"
recname = "rec5"
savedir = "/bucket/ReiterU/OLdev/analysis/20250528_IGL_monitor_flip_analysis/20250521_rec5"

templatefile = os.path.join(rootdir, "2P", recname, "run2/mcorr_nonrigid/QC/QC_nonrigid/template.tif")
roifile = os.path.join(rootdir, "2P", recname, "ROI_transformed_corrected_renamed.zip")
dfffile = os.path.join(rootdir, "2P", recname, "dff_ROI_transformed_corrected_renamed/DFF.hdf5")
syncfile = os.path.join(rootdir, "sync", f"{recname}.tdms")

QI_THRESH = 0.25

### Import stuff

In [4]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from scipy import signal
import h5py
import subprocess

In [5]:
import sys
sys.path.append("../../../")
from calfuncs.io import read_sync_ttl
from calfuncs.full_field_step import get_ett2, plot_chirp_eta, compute_qi

In [6]:
from matplotlib import font_manager
import matplotlib as mpl

font_manager.fontManager.addfont('/bucket/ReiterU/common/fonts/arial.ttf')
mpl.rcParams.update({
    'font.size': 7,
    'font.family': 'sans-serif',
    'font.sans-serif': "Arial",
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
    'axes.linewidth': 0.5,
    'xtick.major.width': 0.5,
    'xtick.major.size': 2,
    'xtick.minor.size': 2,
    'xtick.minor.width': 0.5,
    'ytick.major.width': 0.5,
    'ytick.minor.width': 0.5,
    'ytick.major.size': 2,
    'ytick.minor.size': 2,
})

### Load TDMS file

In [7]:
sync_ttl = read_sync_ttl(syncfile)
twop_ttl = sync_ttl["twop_ttl"]
twop_re = sync_ttl["twop_re"]
vis_re = sync_ttl["vis_re"]
print(twop_ttl.shape, twop_re.shape, vis_re.shape)

(719500,) (10951,) (60,)


### Load DF/F

In [8]:
with h5py.File(dfffile, "r") as hf:
    F = hf["F"][:]
    dff = hf["dff"][:]
print(F.shape) # F is sampled at 1 kH

(190, 719500)


### Chirp

In [9]:
E = []
for nid in np.arange(0, F.shape[0]):
    ett1 = get_ett2(F, nid, vis_re[0::6], window=[-2000, 19000], baseline_window=[-4000, 70000], subsampling=50)
    ett1["session"] = 0
    ett2 = get_ett2(F, nid, vis_re[2::6], window=[-2000, 19000], baseline_window=[-27000, 43000], subsampling=50)
    ett2["session"] = 1
    ett3 = get_ett2(F, nid, vis_re[4::6], window=[-2000, 19000], baseline_window=[-51000, 19000], subsampling=50)
    ett3["session"] = 2
    ett = pd.concat([ett1, ett2, ett3])
    ett["neuron"] = nid
    ett = ett.set_index(["neuron", "session", "trial", "time"])
    E.append(ett)
E = pd.concat(E)

In [ ]:
dt = 1e-2
stimulus = np.zeros(int(21.0/dt), dtype=np.float32)
stimulus[0:int(2.0/dt)] = 0
stimulus[int(2.0/dt):int(6.0/dt)] = 1
stimulus[int(6.0/dt):int(10.0/dt)] = 0
stimulus[int(10.0/dt):int(12.0/dt)] = 0.5
stimulus[int(12.0/dt):int(20.0/dt)] = 0.5 + 0.5 * signal.chirp(np.linspace(0, 8, int(20.0/dt)-int(12.0/dt)), f0=0.5, f1=10, t1=8, method='logarithmic', phi=90)
stimulus[int(20.0/dt):int(21.0/dt)] = 0

for nid in np.arange(0, dff.shape[0]):
    qi = compute_qi(E, nid)
    print("Neuron:", nid, "QI:", qi)
    savename = os.path.join(tmpdir, f"ETA_{nid:03d}.png")
    plot_chirp_eta(E, nid, stimulus, vmin=-2, vmax=3,  cmap_name="bone_r", figsize=(5,2.0), show_plot=True, figsavename=savename)

### Compute trial-average

In [11]:
R = []
neurons = E.index.get_level_values("neuron").unique()

for n in neurons:
    M = []
    df2 = E.xs(n, level="neuron")
    for i in range(3):
        df3 = df2.xs(i, level="session")
        mat = df3.reset_index().pivot(index="trial", columns="time", values="y").to_numpy()
        M.append(mat)
    M = np.hstack(M)
    M = M[~np.all(np.isnan(M), axis=1)] # remove trials with NaN values
    qi = compute_qi(E, n)
    y_mean = np.nanmean(M, axis=0)
    row = {
        "recording": 0,
        "neuron": n,
        "qi": qi,
    }
    for j, value in enumerate(y_mean):
        row[f"time_{j}"] = value

    R.append(row)

R = pd.DataFrame(R)

In [12]:
display(R)

,recording,neuron,qi,time_0,time_1,time_2,time_3,time_4,time_5,time_6,...,time_1250,time_1251,time_1252,time_1253,time_1254,time_1255,time_1256,time_1257,time_1258,time_1259
0,0,0,0.271455,0.830594,1.168060,1.149453,0.719556,0.941190,1.001753,0.725750,...,-0.045674,-0.248594,-0.474544,-0.875681,-0.783215,-0.533228,-0.594334,-0.653696,-0.618699,-0.659304
1,0,1,0.413042,0.878354,0.853189,0.772113,0.938033,0.903208,1.116027,1.034865,...,-1.258365,-1.155760,-1.231541,-1.387791,-1.067376,-1.019863,-1.069732,-0.935813,-1.074359,-0.825753
2,0,2,0.720885,-0.286248,-0.151515,-0.038565,-0.226335,-0.398073,-0.310711,-0.486130,...,1.060645,1.157335,1.175491,0.969906,0.786772,0.858694,0.926834,0.837704,0.914228,0.807051
3,0,3,0.552409,-1.271336,-1.446572,-1.262199,-1.352294,-1.251494,-1.423302,-1.386213,...,-1.005990,-1.140410,-1.297701,-1.064373,-1.102581,-1.015755,-0.870706,-1.229104,-1.320483,-1.106060
4,0,4,0.287313,0.485478,0.231469,0.188571,0.061872,0.162153,0.666646,0.008147,...,-0.186160,-0.409227,0.177254,-0.028535,-0.810891,-0.654773,-0.329395,-0.343952,-0.523771,-0.086544
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
185,0,185,0.128271,-0.161370,-0.068780,-0.189371,-0.008869,-0.463254,-0.419072,-0.936516,...,-0.240931,-0.244121,-0.248674,0.150655,-0.315674,-0.599540,-0.871043,-0.152291,0.093048,-0.192109
186,0,186,0.232516,-0.700132,-0.408042,-0.113345,-0.184829,-0.574938,-0.734765,-0.639991,...,0.059077,0.152632,-0.019785,-0.212788,-0.086720,-0.255985,-0.342789,0.014928,-0.136853,-0.368987
187,0,187,0.331569,0.351195,0.685955,0.644920,-0.060391,-0.046438,0.069526,0.202284,...,-0.744471,-0.971108,-0.505900,-0.424546,-0.657415,-0.917965,-0.916199,-0.677668,-0.946448,-0.873025
188,0,188,0.109702,0.217366,0.192700,0.684099,0.328258,-0.107632,0.201179,0.002635,...,-0.061275,0.389811,0.116371,-0.097065,-0.203761,0.124939,0.192631,-0.279797,-0.463716,-0.299564


In [13]:
with pd.HDFStore(os.path.join(tmpdir, "response_table.hdf5"), mode='w') as store:
    store.put('E', E)
    store.put('responses', R)

### Copy data to bucket

In [14]:
cmd1 = f'ssh deigo "mkdir -p {savedir}"'
cmd2 = f'ssh deigo "cp -r {tmpdir}/* {savedir}"'
print(cmd1)
print(cmd2)

subprocess.run(cmd1, shell=True, check=True)
subprocess.run(cmd2, shell=True, check=True)

ssh deigo "mkdir -p /bucket/ReiterU/OLdev/analysis/20250528_IGL_monitor_flip_analysis/20250521_rec5"
ssh deigo "cp -r /flash/ReiterU/tmp/tomo_LVFLQLRVBQBF/* /bucket/ReiterU/OLdev/analysis/20250528_IGL_monitor_flip_analysis/20250521_rec5"


CompletedProcess(args='ssh deigo "cp -r /flash/ReiterU/tmp/tomo_LVFLQLRVBQBF/* /bucket/ReiterU/OLdev/analysis/20250528_IGL_monitor_flip_analysis/20250521_rec5"', returncode=0)